# 实验 1：从零开始实现可自编辑记忆在 LLM 代理中启用长期记忆有多种方式，例如 RAG 和递归摘要。MemGPT 论文首次提出了 *自编辑记忆* 的概念。本质上，就是把记忆管理交给 LLM。毕竟，LLM 是我们程序中最“智能”的部分，那为什么不让 LLM 自己想办法管理记忆，而是去硬编码某种方案呢？在本节中，我们将演示如何使用 OpenAI 的工具调用来实现一些简单的记忆管理工具。

## 准备工作<div style="background-color:#fff6ff; padding:13px; border-width:3px; border-color:#efe6ef; border-style:solid; border-radius:6px"><p> 💻 &nbsp; <b>访问 <code>requirements.txt</code> 和 <code>helper.py</code> 文件：</b> 1) 点击笔记本顶部菜单中的 <em>"File"</em> 选项，然后 2) 点击 <em>"Open"</em>。<p> ⬇ &nbsp; <b>下载笔记本：</b> 1) 点击笔记本顶部菜单中的 <em>"File"</em> 选项，然后 2) 点击 <em>"Download as"</em> 并选择 <em>"Notebook (.ipynb)"</em>。</p><p> 📒 &nbsp; 如需更多帮助，请参阅 <em>"附录 – 提示、帮助和下载"</em> 课程。</p></div>

<p style="background-color:#f7fff8; padding:15px; border-width:3px; border-color:#e0f0e0; border-style:solid; border-radius:6px"> 🚨&nbsp; <b>不同的运行结果：</b> 由于 AI 模型具有动态且带有概率性的特征，每次执行生成的输出都可能不同。你的结果可能会与视频中展示的内容不一致。</p>

## 第 0 部分：设置 OpenAI

In [1]:
from helper import get_openai_api_key
openai_api_key = get_openai_api_key()

In [2]:
from openai import OpenAI
import os

client = OpenAI(
    api_key=openai_api_key
)

## 第 1 部分：理解 LLM 的上下文窗口LLM 拥有一个 *上下文窗口*（即输入到模型中的 token 集合），而这个窗口的大小是有限的。这意味着我们必须谨慎决定把什么内容放进上下文。通常，代理的上下文窗口会按照某种结构组织起来。不同的代理框架结构会有所不同，但一般都会包含：* 一个用于指示代理行为的 *系统提示** 以往对话的 *聊天历史*由于上下文窗口有限，只有部分聊天历史能够被包含进去。一些框架还会在上下文中加入递归摘要，或者从外部数据库中检索相关消息并放入上下文。在 MemGPT 中，我们还会额外为以下内容预留上下文区域：* 所有历史对话的 *递归摘要** 一个可由代理读写的 *核心记忆* 区域

### 一个简单代理的上下文窗口在下面的代码中，你可以看到如何用系统提示来定义一个代理。系统提示会作为第一条消息包含在每一次聊天完成请求中，而后续消息会随着用户和助手的交互而不断变化。

In [3]:
model = "gpt-4o-mini"

In [4]:
system_prompt = "You are a chatbot."

In [5]:
# 发起一次带工具调用的 completion 请求chat_completion = client.chat.completions.create(    model=model,    messages=[        # 系统提示：始终包含在上下文窗口中        {"role": "system", "content": system_prompt},         # 聊天历史（会随着时间演进）        {"role": "user", "content": "What is my name?"},     ])chat_completion.choices[0].message.content

"I don't have access to personal information about users unless you share it with me in the conversation. If you'd like me to address you by your name, feel free to tell me!"

### 将记忆加入上下文就像我们总是以系统提示开始一次聊天完成请求一样，我们也可以在消息列表前添加一个包含重要记忆的提示。下面看看如何把记忆部分加入上下文窗口，并让代理利用这段记忆来回应用户消息。

In [6]:
agent_memory = {"human": "Name: Bob"}
system_prompt = "You are a chatbot. " \
+ "You have a section of your context called [MEMORY] " \
+ "that contains information relevant to your conversation"

In [7]:
import jsonchat_completion = client.chat.completions.create(    model=model,    messages=[        # 系统提示        {"role": "system", "content": system_prompt + "[MEMORY]" + json.dumps(agent_memory)},        # 聊天历史        {"role": "user", "content": "What is my name?"},    ],)chat_completion.choices[0].message.content

'Your name is Bob.'

在下一节中，我们将演示如何让这段记忆变成代理可读写的内容。

## 第 2 部分：使用工具修改记忆

首先，我们需要定义一个记忆保存工具。为了让代理能够保存新的记忆，我们会实现一个简单的工具，把内容追加到传给代理的记忆字典中的某个部分。

### 定义记忆编辑工具我们不会直接把名字写进代理的记忆里，而是从一个空的记忆对象开始，并提供一个能够编辑该记忆对象的函数。

In [8]:
agent_memory = {"human": "", "agent": ""}

def core_memory_save(section: str, memory: str): 
    agent_memory[section] += '\n' 
    agent_memory[section] += memory 

为了让代理知道这个工具以及如何使用它，我们需要创建一个 OpenAI 可以处理的工具 schema。这包括工具的用途说明，以及代理在调用工具时必须生成的参数。

In [9]:
# 工具说明core_memory_save_description = "Save important information about you," + "the agent or the human you are chatting with."# 工具的参数（由 LLM 生成）# 这里定义代理必须生成哪些内容才能调用该工具core_memory_save_properties = {    # 参数 1：要编辑的记忆分区    "section": {        "type": "string",        "enum": ["human", "agent"],        "description": "Must be either 'human' "         + "(to save information about the human) or 'agent'"         + "(to save information about yourself)",                },    # 参数 2：要保存的记忆内容    "memory": {        "type": "string",        "description": "Memory to save in the section",    },}# 工具 schema（传递给 OpenAI）core_memory_save_metadata =     {        "type": "function",        "function": {            "name": "core_memory_save",            "description": core_memory_save_description,            "parameters": {                "type": "object",                "properties": core_memory_save_properties,                "required": ["section", "memory"],            },        }    }

现在，我们可以把工具调用传给代理了！

In [10]:
agent_memory = {"human": ""}system_prompt = "You are a chatbot. " + "You have a section of your context called [MEMORY] " + "that contains information relevant to your conversation"chat_completion = client.chat.completions.create(    model=model,    messages=[        # 系统提示        {"role": "system", "content": system_prompt},         # 记忆        {"role": "system", "content": "[MEMORY]" + json.dumps(agent_memory)},        # 聊天历史        {"role": "user", "content": "My name is Bob"},    ],    # 工具 schema    tools=[core_memory_save_metadata])response = chat_completion.choices[0]response

Choice(finish_reason='tool_calls', index=0, logprobs=None, message=ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=[ChatCompletionMessageToolCall(id='call_AKzJVlIIfdeUnZlEzVdaHe7P', function=Function(arguments='{"section":"human","memory":"Human\'s name is Bob."}', name='core_memory_save'), type='function')]))

### 执行工具遗憾的是，OpenAI 并不会真正执行这个工具，所以这一步需要我们自己来完成。下面我们用工具调用响应中给出的参数来运行这个工具。

In [11]:
arguments = json.loads(response.message.tool_calls[0].function.arguments)
arguments

{'section': 'human', 'memory': "Human's name is Bob."}

In [12]:
# run the function with the specified arguments 
core_memory_save(**arguments)

现在，我们可以查看已经更新后的记忆对象：

In [13]:
agent_memory

{'human': "\nHuman's name is Bob."}

### 运行代理的下一步现在，我们可以观察到，在记忆更新之后，代理的响应会有什么不同。

In [14]:
chat_completion = client.chat.completions.create(    model=model,    messages=[        # 系统提示        {"role": "system", "content": system_prompt},         # 记忆        {"role": "system", "content": "[MEMORY]" + json.dumps(agent_memory)},        # 聊天历史        {"role": "user", "content": "What is my name?"},    ],    # 工具 schema    tools=[core_memory_save_metadata])response = chat_completion.choices[0]response

ChatCompletionMessage(content='Your name is Bob.', refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=None)

恭喜！你已经实现了一个记忆编辑函数。你能想到怎样为其他数据结构也加入支持吗？

## 实现代理循环在当前的实现中，代理一次只能执行一个步骤：它要么编辑记忆，要么回复用户。但理想情况下，我们希望代理支持 *多步推理*，这样它就能把多个动作串联起来。例如，如果我们告诉代理我们的名字是 “Bob”，我们希望它既能更新记忆，也能返回一条回复消息。我们可以通过在循环中调用 `client.chat.completions.create(...)` 来实现多步推理，并允许代理自行决定是继续推理，还是跳出循环。为简化起见，我们假设任何*不是*工具调用的代理响应都会跳出推理循环，并把控制权交还给用户。

In [15]:
agent_memory = {"human": ""}

In [16]:
system_prompt_os = system_prompt \
+ "\n. You must either call a tool (core_memory_save) or" \
+ "write a response to the user. " \
+ "Do not take the same actions multiple times!" \
+ "When you learn new information, make sure to always" \
+ "call the core_memory_save tool." 

我们为代理实现了一个简单的 step 函数。这个函数会响应用户消息，但也允许代理按顺序执行多个动作，并在发送一条不调用工具的消息时返回给用户。

In [17]:
def agent_step(user_message, chat_history = []):     # 在消息前补上系统提示和记忆    messages = [        # 系统提示        {"role": "system", "content": system_prompt_os},         # 记忆        {            "role": "system",             "content": "[MEMORY]" + json.dumps(agent_memory)        },    ]     # 追加聊天历史    messages += chat_history        # 追加最新消息    messages.append({"role": "user", "content": user_message})        # 代理循环    while True:         chat_completion = client.chat.completions.create(            model=model,            messages=messages,            tools=[core_memory_save_metadata]        )        response = chat_completion.choices[0]        # 用代理的响应更新消息列表        messages.append(response.message)        # 如果没有调用工具（即直接回复用户），就返回        if not response.message.tool_calls:             messages.append({                "role": "assistant",                 "content": response.message.content            })            return response.message.content        # 如果调用了工具，就执行工具        if response.message.tool_calls:             print("TOOL CALL:", response.message.tool_calls[0].function)            # 把工具调用结果加入消息历史            messages.append({"role": "tool", "tool_call_id": response.message.tool_calls[0].id, "name": "core_memory_save", "content": f"Updated memory: {json.dumps(agent_memory)}"})            # 从 LLM 的函数调用中解析参数            arguments = json.loads(response.message.tool_calls[0].function.arguments)            # 使用指定参数运行函数            core_memory_save(**arguments)

现在，我们可以观察到，当我们向代理发送一条消息时，它会执行多个动作：

In [18]:
# 动作 1（幕后）：AI 发现用户说了名字，触发工具调用 core_memory_save，把 "Bob" 存入字典。
# 动作 2（台前）：AI 在第二次循环中看到记忆里已经有 Bob 了，于是回复：'Nice to meet you, Bob! How can I assist you today?'
agent_step("my name is bob.")

TOOL CALL: Function(arguments='{"section":"human","memory":"The user\'s name is Bob."}', name='core_memory_save')


'Nice to meet you, Bob! How can I assist you today?'

现在，代理已经能够在一个步骤中同时编辑自己的记忆，*并且* 生成一条利用更新后记忆的用户回复。虽然在这个示例中，我们只支持一个工具以及向用户回复这两种动作，但同样的结构可以扩展为更复杂的推理循环，并组合许多不同工具。在 MemGPT 中，所有动作（甚至包括给用户发消息）都被建模为工具；其中一些工具（例如发送消息）会中断代理的推理循环，而另一些工具（例如搜索归档记忆存储或编辑记忆）则不会。恭喜！你已经实现了一个带有自编辑记忆和多步推理能力的代理 :)